# Create v4 (smaller design matrix)

Builds `config/{model}/v4` from `v3` (keeps v3 leakage skips + type conversions), then applies feature rules:
- **Raw vs imputed:** every `X_raw` with an `X_imps/_impa/_impm/_imp` partner -> `skip` (keep imputed)
- **`owh_loo_years_cal` -> skip** (48 one-hot columns); `owh_loo_years_imp` (numeric) is kept
- **`dml_make_raw` -> `group_then_ohe:0.01`** (makes < 1% of train rows -> `__OTHER__`)
- Constant columns and other one-hot columns unchanged

Load list is rebuilt: v1 rows unchanged + enabled master columns (skipped columns are not loaded).
Runners are created in `runners/v4/`. Best params are copied from v3 (05b off).

In [ ]:
src_version = "v3"
dst_version = "v4"
base_version = "v1"   # curated load list kept unchanged
models = None          # None = all config/*/{src_version}; or e.g. ["car_coll"]
overwrite = False      # True = rebuild existing v4 folders

# Feature rules
drop_raw_with_imputed = True
skip = ["owh_loo_years_cal"]
group_ohe = {"dml_make_raw": 0.01}
to_numeric = []

In [ ]:
import sys
from pathlib import Path

project_root = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / 'current.pc').exists())
sys.path.insert(0, str(project_root / 'lib'))
from utils import setup_notebook_environment, load_config
from version_builder import (build_full_feature_version, apply_feature_rules,
                             check_full_feature_version, create_version_runner, _aux_columns)

project_root = setup_notebook_environment()
if models is None:
    models = sorted(p.parent.parent.name for p in Path('config').glob(f'*_*/{src_version}/config.yaml'))
print(f"{len(models)} models: {models}")

In [ ]:
# Build v4 from v3, apply rules, create runner, validate
for m in models:
    build_full_feature_version(project_root, m, src_version, dst_version, overwrite=overwrite, verbose=False)
    apply_feature_rules(project_root, m, dst_version, base_version=base_version,
                        drop_raw_with_imputed=drop_raw_with_imputed, skip=skip,
                        group_ohe=group_ohe, to_numeric=to_numeric)
    create_version_runner(project_root, m, src_version, dst_version)
    aux = _aux_columns(load_config(f'config/{m}/{base_version}/config.yaml'), Path(f'config/{m}/{base_version}'))
    problems = check_full_feature_version(project_root, m, src_version, dst_version, aux_columns=aux,
                                          verbose_excluded=False, base_version=base_version)
    print(f"   check: {problems or 'OK'}")